# Filter India RDD → `data/road`

Colab CPU / Python 3. Run the **one** code cell below from the top (do not start halfway).

Writes India files to `/content/drive/MyDrive/data/road`.

In [ ]:
from pathlib import Path
import shutil
from collections import Counter
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
KEEP_CLASSES = {0, 1, 2, 3}
COUNTRY_PREFIX = "India_"
SPLITS = ("train", "val", "test")
RDD_DRIVE_URL = "https://drive.google.com/drive/folders/16aHH0rPP5-0BSoC3LV_5yj0AT5r46Ed3?usp=sharing"
DEST = Path("/content/drive/MyDrive/data/road")


def is_rdd_split(path):
    return path.is_dir() and all((path / split).is_dir() for split in SPLITS)


def walk_named(root, name, max_depth=3):
    if not root.exists():
        return None
    stack = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == name and is_rdd_split(current):
            return current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if child.is_dir():
                stack.append((child, depth + 1))
    return None


def find_rdd_split():
    for path in [
        Path("/content/RDD_SPLIT"),
        Path("/content/drive/MyDrive/RDD_SPLIT"),
        Path("/content/drive/MyDrive/datasets/RDD_SPLIT"),
        Path("/content/drive/MyDrive/datasets/archive/RDD_SPLIT"),
    ]:
        if is_rdd_split(path):
            return path
    for root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")):
        found = walk_named(root, "RDD_SPLIT")
        if found is not None:
            return found
    return None


def is_india(path):
    return path.name.startswith(COUNTRY_PREFIX) or path.name.lower().startswith("india_")


def filter_label_text(lbl):
    kept, dropped, lines = Counter(), Counter(), []
    if not lbl.exists():
        return "", kept, dropped
    for raw in lbl.read_text(encoding="utf-8").splitlines():
        if not raw.strip():
            continue
        parts = raw.split()
        if len(parts) != 5:
            continue
        cls = int(float(parts[0]))
        if cls not in KEEP_CLASSES:
            dropped[cls] += 1
            continue
        lines.append(" ".join(parts[:5]))
        kept[cls] += 1
    text = "\n".join(lines) + ("\n" if lines else "")
    return text, kept, dropped


src = find_rdd_split()
assert src is not None, (
    "RDD_SPLIT not found on My Drive. Put the shortcut at My Drive/RDD_SPLIT.\n"
    + RDD_DRIVE_URL
)
print("RDD_SPLIT:", src)
print("Writing India files to:", DEST)

stats = {split: {"images": 0, "boxes": Counter(), "dropped": Counter()} for split in SPLITS}
for split in SPLITS:
    src_img = src / split / "images"
    src_lbl = src / split / "labels"
    dst_img = DEST / split / "images"
    dst_lbl = DEST / split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    if not src_img.exists():
        print("missing", src_img)
        continue
    files = [p for p in src_img.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    print(f"{split}: scanning {len(files)} images...")
    for i, img in enumerate(files, 1):
        if not is_india(img):
            continue
        text, kept, dropped = filter_label_text(src_lbl / f"{img.stem}.txt")
        dst = dst_img / img.name
        if not (dst.exists() and dst.stat().st_size == img.stat().st_size):
            shutil.copy2(img, dst)
        (dst_lbl / f"{img.stem}.txt").write_text(text, encoding="utf-8")
        stats[split]["images"] += 1
        stats[split]["boxes"].update(kept)
        stats[split]["dropped"].update(dropped)
        if i % 500 == 0:
            print(f"  {split} scanned {i}/{len(files)}, kept {stats[split]['images']}")

names = {
    0: "D00_longitudinal_crack",
    1: "D10_transverse_crack",
    2: "D20_alligator_crack",
    3: "D40_pothole",
}
(DEST / "data.yaml").write_text(
    f"path: {DEST.as_posix()}\n"
    "train: train/images\n"
    "val: val/images\n"
    "test: test/images\n"
    "nc: 4\n"
    "names:\n"
    f"  0: {names[0]}\n"
    f"  1: {names[1]}\n"
    f"  2: {names[2]}\n"
    f"  3: {names[3]}\n",
    encoding="utf-8",
)

for split, s in stats.items():
    print(f"\n=== {split} (India only) ===")
    print("images:", s["images"])
    print("boxes:", dict(s["boxes"]))
    print("dropped non-target classes:", dict(s["dropped"]))

if stats["train"]["images"] == 0:
    train_img = src / "train" / "images"
    sample = [p.name for p in list(train_img.iterdir())[:8] if p.is_file()] if train_img.exists() else []
    raise SystemExit(f"No India images with prefix '{COUNTRY_PREFIX}'. Sample: {sample}")

print("\nDone:", DEST)